# Solução comentada — Simulado "Festival ViraBairro"

As 8 questões resolvidas, usando os nomes de coluna dos enunciados.
Serve como **exemplo de como adaptar** os templates dos arquivos `.md`.

**Como usar na prova:**
- cada questão tem o código e a resposta escrita logo abaixo;
- as células imprimem os **números que você precisa** para preencher a
  resposta (marcados como `>>> PARA A RESPOSTA`);
- onde a resposta depende do resultado, está marcado com `[CONFIRMAR]` —
  troque pelo valor que saiu no seu notebook.

O código foi testado contra uma base sintética com o mesmo esquema de
colunas (pandas 1.5, 2.2 e 3.0). Os **números** que aparecem na sua execução serão os reais — as
frases marcadas com `[CONFIRMAR]` precisam ser ajustadas a eles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

pd.set_option("display.max_columns", None)

FONTE = "Fonte: dados sintéticos do Festival ViraBairro (2026)"
def fonte():
    plt.figtext(0.5, -0.05, FONTE, ha="center", fontsize=9, style="italic")

# características disponíveis ANTES da publicação (usadas nas Q6, Q7 e Q8)
FEATURES = ["tema", "formato", "seguidores_autor", "videos_autor",
            "tamanho_legenda", "n_emojis", "n_hashtags", "hora",
            "dia_semana", "duracao_segundos"]

---
## Questão 1 — Diagnóstico inicial da base

In [ ]:
df1 = pd.read_csv("dados/publicacoes_brutas.csv")

# 1. cinco primeiras linhas
print(df1.head())

# 2. dimensões
print(f"\nLinhas: {df1.shape[0]}, Colunas: {df1.shape[1]}")

# 3. ausências — somente colunas com ao menos uma
ausencias = df1.isna().sum()
print("\nAusências por coluna:")
print(ausencias[ausencias > 0])

# >>> PARA A RESPOSTA: contexto do recorte
# (a data ainda é texto aqui, então min/max não faria sentido — só na Q2, já convertida)
print("\n>>> PARA A RESPOSTA")
print("total de registros:", len(df1))
print("grafias distintas de tema (antes de padronizar):", df1["tema"].nunique())
print("formatos:", df1["formato"].unique())
print("duplicatas por id_publicacao:", df1.duplicated(subset="id_publicacao").sum())

**Resposta da Questão 1: limite de uso:**

A base cobre apenas as publicações de um único festival fictício, em um
recorte de oito semanas e em um conjunto restrito de perfis, e os dados são
sintéticos. Por isso os resultados descrevem somente esse conjunto de
registros e não podem ser tratados como retrato de todas as redes, públicos
ou bairros.

---
## Questão 2 — Tratamento dos registros

Atenção à conversão de datas: a base bruta tem formatos misturados. O jeito
simples (`errors="coerce"` sozinho) descarta calado um dos formatos, e o jeito
da Aula 10 (`format="mixed", dayfirst=True`) inverte dia e mês das datas ISO no
pandas 3. Por isso o `converter_data` (explicado no `02-limpeza.md`).

In [ ]:
df2 = pd.read_csv("dados/publicacoes_brutas.csv")
print("antes:", df2.shape)

# 1. remover duplicidade por id_publicacao
df2 = df2.drop_duplicates(subset="id_publicacao").copy()
print("após remover duplicatas:", df2.shape)

# 2. padronizar tema
print("temas antes:", sorted(df2["tema"].dropna().unique()))
df2["tema"] = df2["tema"].str.strip().str.lower().str.title()
print("temas depois:", sorted(df2["tema"].dropna().unique()))

# 3. converter tipos
PANDAS_2 = int(pd.__version__.split(".")[0]) >= 2

def converter_data(serie):
    texto = serie.astype("string").str.strip()
    eh_iso = texto.str.match(r"\d{4}-\d{2}-\d{2}").fillna(False).astype(bool)   # começa com AAAA-MM-DD
    if PANDAS_2:
        iso = pd.to_datetime(texto.where(eh_iso), format="ISO8601", errors="coerce")
        outros = pd.to_datetime(texto.where(~eh_iso), format="mixed", dayfirst=True, errors="coerce")
    else:
        iso = pd.to_datetime(texto.where(eh_iso), errors="coerce")
        outros = pd.to_datetime(texto.where(~eh_iso), dayfirst=True, errors="coerce")
    return iso.fillna(outros)

df2["data_publicacao"] = converter_data(df2["data_publicacao"])
print("datas não convertidas:", df2["data_publicacao"].isna().sum())
print("período:", df2["data_publicacao"].min(), "→", df2["data_publicacao"].max())

for col in ["compartilhamentos", "salvamentos", "alcance"]:
    df2[col] = pd.to_numeric(df2[col], errors="coerce")

# 4. ausências e inválidos
df2 = df2[df2["alcance"] > 0].copy()        # descarta NaN, zero e negativo do denominador
df2["compartilhamentos"] = df2["compartilhamentos"].fillna(df2["compartilhamentos"].median())
df2["salvamentos"] = df2["salvamentos"].fillna(df2["salvamentos"].median())
print("após tratar ausências/inválidos:", df2.shape)

# 5. taxa_utilidade_pct
df2["taxa_utilidade_pct"] = (
    (df2["compartilhamentos"] + df2["salvamentos"]) / df2["alcance"] * 100
)

# tabela por tema, ordenada pela mediana
resumo_tema = (
    df2.groupby("tema")["taxa_utilidade_pct"]
    .agg(publicacoes="count", mediana="median")
    .sort_values("mediana", ascending=False)
)
print("\n", resumo_tema)

**Resposta da Questão 2: decisões de tratamento:**

Removi os registros duplicados por `id_publicacao` para não contar a mesma
publicação duas vezes, e padronizei `tema` (espaços e caixa) porque a mesma
categoria aparecia escrita de formas diferentes, o que dividiria os grupos.
Converti `data_publicacao` tratando separadamente os dois formatos presentes
na base, para não descartar nem inverter registros, e converti as colunas
numéricas do cálculo com `to_numeric`. Descartei as linhas com `alcance`
ausente, zerado ou negativo, já que sem um denominador válido a taxa não pode
ser calculada. Preenchi as ausências de `compartilhamentos` e `salvamentos`
com a mediana de cada coluna, por ser menos sensível a valores extremos que a
média e por preservar as linhas restantes.

---
## Questão 3 — Escolha de tema para divulgação

In [ ]:
df3 = pd.read_csv("dados/publicacoes_analise.csv")

df3["taxa_utilidade_pct"] = (
    (df3["compartilhamentos"] + df3["salvamentos"]) / df3["alcance"] * 100
)

# 1. tabela com a mediana por tema
mediana_tema = (df3.groupby("tema")["taxa_utilidade_pct"]
                .median().sort_values(ascending=False))
print(mediana_tema.round(2))

# 2. gráfico de barras
plt.figure(figsize=(8, 5))
plt.bar(mediana_tema.index, mediana_tema.values, color="#4C72B0")
plt.title("Qual tema as pessoas mais salvam ou compartilham?")
plt.xlabel("Tema")
plt.ylabel("Mediana da taxa de utilidade (%)")
fonte()
plt.tight_layout()
plt.show()

print(">>> PARA A RESPOSTA")
print("tema com maior mediana:", mediana_tema.index[0], "=", round(mediana_tema.iloc[0], 2), "%")
print("tema com menor mediana:", mediana_tema.index[-1], "=", round(mediana_tema.iloc[-1], 2), "%")
print(df3["tema"].value_counts())

**Resposta da Questão 3: recomendação para a equipe:**

Recomendo dar o destaque principal ao tema **[CONFIRMAR: tema com maior
mediana]**, que apresentou a maior mediana de taxa de utilidade
(**[CONFIRMAR: valor]%**) entre os quatro temas. A mediana representa o valor
típico da taxa entre as publicações daquele tema — metade ficou acima e
metade abaixo desse valor — e é menos sensível a publicações excepcionais do
que a média, o que evita que um único pico distorça a leitura. Como a taxa
considera compartilhamentos e salvamentos, ela indica conteúdo que as pessoas
guardam ou repassam, e não apenas o que alcança mais gente. Uma limitação é
que os temas também diferem em formato, horário e perfil de autor, de modo
que a diferença observada não isola o efeito do tema. A escolha é editorial e
apoiada em evidência descritiva: não afirma que o tema cause maior
compartilhamento.

---
## Questão 4 — Acompanhamento diário

In [ ]:
df4 = pd.read_csv("dados/publicacoes_analise.csv")

# 1. converter data e criar dia_publicacao
df4["data_publicacao"] = pd.to_datetime(df4["data_publicacao"], errors="coerce")
print("datas não convertidas:", df4["data_publicacao"].isna().sum())
df4["dia_publicacao"] = df4["data_publicacao"].dt.date

# 2. tabela por dia, ordenada cronologicamente
diario = (
    df4.groupby("dia_publicacao")
    .agg(publicacoes=("id_publicacao", "count"),
         engajamento_medio=("taxa_engajamento_pct", "mean"),
         alcance_total=("alcance", "sum"))
    .sort_index()
)
print(diario.round(2))

# 3. gráfico de linhas
plt.figure(figsize=(10, 5))
plt.plot(diario.index, diario["engajamento_medio"], marker="o", color="#4C72B0")
plt.title("Taxa média de engajamento por dia da campanha")
plt.xlabel("Dia da publicação")
plt.ylabel("Taxa média de engajamento (%)")
plt.xticks(rotation=45, ha="right")
fonte()
plt.tight_layout()
plt.show()

# 4. recorte: reels a partir das 18h
recorte = df4[(df4["formato"] == "reel") & (df4["hora"] >= 18)]
print("publicações no recorte:", len(recorte))

top5 = recorte.nlargest(5, "taxa_engajamento_pct")[
    ["id_publicacao", "dia_publicacao", "hora", "tema", "taxa_engajamento_pct"]
]
print(top5)

print("\n>>> PARA A RESPOSTA")
print("dias cobertos:", diario.index.min(), "a", diario.index.max(), f"({len(diario)} dias)")
print("menor média diária:", round(diario['engajamento_medio'].min(), 2))
print("maior média diária:", round(diario['engajamento_medio'].max(), 2))

**Resposta da Questão 4: leitura da agenda:**

A taxa média de engajamento oscila entre os dias da campanha, variando de
**[CONFIRMAR: menor]%** a **[CONFIRMAR: maior]%**, com picos pontuais e sem um
padrão consistente de alta ou de queda ao longo do período. O intervalo
coberto é curto e o número de publicações por dia é pequeno e desigual, o que
torna a média diária instável — não é possível caracterizar tendência de
longo prazo. O recorte de reels publicados a partir das 18h reúne peças com
engajamento alto, mas isso não demonstra que o horário ou o formato causem
esse resultado: essas publicações também diferem em tema, tamanho de legenda
e perfil de autor, e não houve comparação controlada entre horários e
formatos.

---
## Questão 5 — Tema e formato das publicações

In [ ]:
df5 = pd.read_csv("dados/publicacoes_analise.csv")

# tabela: uma linha por combinação tema x formato
resumo5 = (
    df5.groupby(["tema", "formato"])["taxa_engajamento_pct"]
    .agg(publicacoes="count", mediana="median")
    .sort_values("mediana", ascending=False)
    .reset_index()
)
print(resumo5.round(2))

# visão em matriz (opcional, ajuda a enxergar o padrão)
print("\n", df5.pivot_table(index="tema", columns="formato",
                            values="taxa_engajamento_pct", aggfunc="median").round(2))

# gráfico de barras comparando as combinações
resumo5 = resumo5.copy()
resumo5["rotulo"] = resumo5["tema"] + " – " + resumo5["formato"]

plt.figure(figsize=(12, 6))
plt.bar(resumo5["rotulo"], resumo5["mediana"], color="#55A868")
plt.title("Mediana da taxa de engajamento por combinação de tema e formato")
plt.xlabel("Combinação tema × formato")
plt.ylabel("Mediana da taxa de engajamento (%)")
plt.xticks(rotation=45, ha="right")
fonte()
plt.tight_layout()
plt.show()

print(">>> PARA A RESPOSTA")
print("melhor combinação:", resumo5.iloc[0]["tema"], "-", resumo5.iloc[0]["formato"],
      "| mediana:", round(resumo5.iloc[0]["mediana"], 2),
      "| n =", int(resumo5.iloc[0]["publicacoes"]))
print("combinações com menos de 10 publicações:",
      (resumo5["publicacoes"] < 10).sum(), "de", len(resumo5))

**Resposta da Questão 5: análise de tema e formato:**

A combinação **[CONFIRMAR: tema – formato]** apresentou a maior mediana de
taxa de engajamento (**[CONFIRMAR: valor]%**) e merece ser testada pela
equipe nas próximas publicações. Comparar duas variáveis ao mesmo tempo é
diferente de analisar cada uma isoladamente porque um tema pode ter desempenho
forte em um formato e fraco em outro: a mediana do tema sozinho resulta de uma
mistura de formatos e pode não corresponder a nenhuma das situações reais.
O cruzamento revela justamente onde o desempenho se concentra, o que é mais
útil para decidir o que produzir do que a média geral do tema. Uma limitação
importante é que várias combinações têm poucas publicações, o que torna a
mediana instável nesses grupos e desaconselha decisões baseadas apenas nelas.
Além disso, a diferença observada não isola o efeito do formato, já que
horário, legenda e perfil do autor variam entre as peças.

---
## Questão 6 — Variáveis mais usadas pela árvore

In [ ]:
df6 = pd.read_csv("dados/publicacoes_analise.csv")

# 1. alvo pelo percentil 75
limite6 = df6["taxa_engajamento_pct"].quantile(0.75)
df6["mereceu_divulgacao_adicional"] = (df6["taxa_engajamento_pct"] > limite6).astype(int)
print("limite p75:", round(limite6, 2))
print(df6["mereceu_divulgacao_adicional"].value_counts())

# 2. características disponíveis antes da publicação
X6 = pd.get_dummies(df6[FEATURES], columns=["tema", "formato"],
                    drop_first=True, dtype=int)
y6 = df6["mereceu_divulgacao_adicional"]

# 3. treino/teste estratificado + árvore
X6_tr, X6_te, y6_tr, y6_te = train_test_split(
    X6, y6, test_size=0.25, random_state=42, stratify=y6)

arvore6 = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced")
arvore6.fit(X6_tr, y6_tr)

# 4. tabela e gráfico das cinco maiores importâncias
importancias6 = pd.Series(arvore6.feature_importances_,
                          index=X6_tr.columns).sort_values(ascending=False)
top5_imp = importancias6.head(5)
print("\n", top5_imp.round(4))

grafico = top5_imp.sort_values()          # crescente: maior fica no topo
plt.figure(figsize=(8, 5))
plt.barh(grafico.index, grafico.values, color="#C44E52")
plt.title("Cinco características mais usadas pela árvore de classificação")
plt.xlabel("Importância (redução de impureza)")
plt.ylabel("Característica")
plt.tight_layout()
plt.show()

print(">>> PARA A RESPOSTA")
print("característica mais importante:", top5_imp.index[0], round(top5_imp.iloc[0], 3))

**Resposta da Questão 6: leitura das importâncias:**

A importância indica apenas o quanto a árvore usou cada característica para
separar as classes **nesta base de treino**, e não demonstra relação causal:
a variável pode estar correlacionada a outro fator não observado, ou ser
escolhida por acaso entre características parecidas entre si. Uma mudança na
composição da base alteraria esse ranking — por exemplo, a entrada de muitas
publicações de um formato novo, uma concentração de peças em um único tema, ou
uma alteração na distribuição orgânica da plataforma que mudasse quais
publicações atingem o percentil 75.

---
## Questão 7 — Estimativa de engajamento

In [ ]:
df7 = pd.read_csv("dados/publicacoes_analise.csv")

# 2. somente características disponíveis antes da publicação (sem vazamento)
X7 = pd.get_dummies(df7[FEATURES], columns=["tema", "formato"],
                    drop_first=True, dtype=int)
y7 = df7["taxa_engajamento_pct"]

# 3. treino/teste
X7_tr, X7_te, y7_tr, y7_te = train_test_split(X7, y7, test_size=0.25, random_state=42)

# 4. regressão linear e árvore de regressão
linear7 = LinearRegression().fit(X7_tr, y7_tr)
arvore7 = DecisionTreeRegressor(max_depth=4, random_state=42).fit(X7_tr, y7_tr)

tabela7 = pd.DataFrame([
    {"modelo": nome,
     "MAE": mean_absolute_error(y7_te, m.predict(X7_te)),
     "R2":  r2_score(y7_te, m.predict(X7_te))}
    for nome, m in [("Regressão linear", linear7), ("Árvore de regressão", arvore7)]
]).sort_values("MAE")
print(tabela7.round(3))

# 5. dispersão real x previsto para o de menor MAE
melhor7 = linear7 if tabela7.iloc[0]["modelo"] == "Regressão linear" else arvore7
y7_prev = melhor7.predict(X7_te)

plt.figure(figsize=(6, 6))
plt.scatter(y7_te, y7_prev, alpha=0.6, color="#4C72B0")
lo = min(y7_te.min(), y7_prev.min()); hi = max(y7_te.max(), y7_prev.max())
plt.plot([lo, hi], [lo, hi], "--", color="gray", label="previsão = valor real")
plt.title(f"Valores reais vs. previstos — {tabela7.iloc[0]['modelo']}")
plt.xlabel("Taxa de engajamento real (%)")
plt.ylabel("Taxa de engajamento prevista (%)")
plt.legend()
plt.tight_layout()
plt.show()

print(">>> PARA A RESPOSTA")
print("modelo escolhido:", tabela7.iloc[0]["modelo"])
print("MAE:", round(tabela7.iloc[0]["MAE"], 2), "| R2:", round(tabela7.iloc[0]["R2"], 3))

**Resposta da Questão 7: estimativa:**

O tipo de aprendizado adequado é a **regressão**, porque a variável-alvo
(`taxa_engajamento_pct`) é numérica e contínua: a equipe quer estimar um
valor esperado, e não atribuir uma categoria (classificação) nem descobrir
grupos sem rótulo prévio (clusterização). O **MAE** mede o erro médio
absoluto entre o valor previsto e o observado, na mesma unidade da taxa — um
MAE de **[CONFIRMAR: valor]** significa que, em média, a estimativa erra essa
quantidade de pontos percentuais. Escolhi a **[CONFIRMAR: modelo de menor
MAE]**, por apresentar o menor erro médio no conjunto de teste. O R² obtido
foi **[CONFIRMAR: valor]**, indicando o quanto da variação do engajamento as
características disponíveis antes da publicação conseguem explicar. Uma
limitação é que o modelo capta apenas associações estatísticas nesta base:
fatores não medidos, como o conteúdo da peça e a distribuição feita pelo
algoritmo da plataforma, podem explicar o resultado, de modo que a previsão
não deve ser interpretada como relação causal.

---
## Questão 8 — Priorização de divulgação

**Resposta da Questão 8.1: tipo de modelo:**

O tipo adequado é a **classificação**, porque o alvo
(`mereceu_divulgacao_adicional`) é categórico: cada publicação recebe o rótulo
1 ou 0. A **regressão** não responde diretamente à decisão porque estimaria um
número contínuo, enquanto a escolha da equipe é binária — dar ou não
divulgação adicional a uma peça. A **clusterização** também não serve porque é
um método não supervisionado: ela agruparia publicações semelhantes sem usar o
rótulo, mas aqui o rótulo já é conhecido na base e o objetivo é prevê-lo para
publicações novas.

In [ ]:
df8 = pd.read_csv("dados/publicacoes_analise.csv")

# alvo: acima do percentil 75
limite8 = df8["taxa_engajamento_pct"].quantile(0.75)
df8["mereceu_divulgacao_adicional"] = (df8["taxa_engajamento_pct"] > limite8).astype(int)
print("limite p75:", round(limite8, 2))
print(df8["mereceu_divulgacao_adicional"].value_counts(), "\n")

# 2. características antes da publicação (sem vazamento: sem alcance,
#    interações, id nem a própria taxa_engajamento_pct)
X8 = pd.get_dummies(df8[FEATURES], columns=["tema", "formato"],
                    drop_first=True, dtype=int)
y8 = df8["mereceu_divulgacao_adicional"]

# 3. treino/teste preservando a proporção do alvo
X8_tr, X8_te, y8_tr, y8_te = train_test_split(
    X8, y8, test_size=0.25, random_state=42, stratify=y8)

# 4. três classificadores da lista
modelos8 = {
    "Regressão logística": make_pipeline(          # sem class_weight, como na Aula 12:
        StandardScaler(),                            # a classe rara é compensada pelo corte (item 6)
        LogisticRegression(max_iter=1000)),
    "Árvore de classificação": DecisionTreeClassifier(
        max_depth=4, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        random_state=42, class_weight="balanced"),
}
for m in modelos8.values():
    m.fit(X8_tr, y8_tr)

# 5. precisão, recall e F1 no teste
tabela8 = pd.DataFrame([
    {"modelo": nome,
     "precisao": precision_score(y8_te, m.predict(X8_te), zero_division=0),
     "recall":   recall_score(y8_te, m.predict(X8_te), zero_division=0),
     "f1":       f1_score(y8_te, m.predict(X8_te), zero_division=0)}
    for nome, m in modelos8.items()
]).sort_values("f1", ascending=False)
print(tabela8.round(3), "\n")

# matriz de confusão do modelo com maior F1
melhor_nome8 = tabela8.iloc[0]["modelo"]
melhor8 = modelos8[melhor_nome8]
cm8 = confusion_matrix(y8_te, melhor8.predict(X8_te))

ConfusionMatrixDisplay(cm8, display_labels=["Não", "Sim"]).plot(cmap="Blues")
plt.title(f"Matriz de confusão — {melhor_nome8}")
plt.show()

vn, fp, fn, vp = cm8.ravel()
print(f"VN={vn}  FP={fp}  FN={fn}  VP={vp}")

# 6. cortes 0,50 e 0,30 na logística já ajustada
prob8 = modelos8["Regressão logística"].predict_proba(X8_te)[:, 1]
tabela_cortes8 = pd.DataFrame([
    {"corte": corte,
     "precisao": precision_score(y8_te, (prob8 >= corte).astype(int), zero_division=0),
     "recall":   recall_score(y8_te, (prob8 >= corte).astype(int), zero_division=0),
     "f1":       f1_score(y8_te, (prob8 >= corte).astype(int), zero_division=0)}
    for corte in [0.50, 0.30]
])
print("\n", tabela_cortes8.round(3))

print("\n>>> PARA A RESPOSTA")
print("modelo com maior F1:", melhor_nome8, "| F1 =", round(tabela8.iloc[0]["f1"], 3))
print("corte 0,50 -> precisão", round(tabela_cortes8.iloc[0]["precisao"], 3),
      "| recall", round(tabela_cortes8.iloc[0]["recall"], 3))
print("corte 0,30 -> precisão", round(tabela_cortes8.iloc[1]["precisao"], 3),
      "| recall", round(tabela_cortes8.iloc[1]["recall"], 3))

**Resposta da Questão 8: decisão e riscos de erro:**

Escolhi o modelo **[CONFIRMAR: maior F1]**, com o corte de **[CONFIRMAR:
0,50 ou 0,30]**. Um **falso positivo** significa destinar um dos poucos
espaços de divulgação a uma publicação que não traria retorno relevante,
desperdiçando um recurso escasso a duas semanas do festival. Um **falso
negativo** significa deixar de impulsionar uma peça que teria bom desempenho,
resultando em oportunidade perdida. Ao baixar o corte de 0,50 para 0,30, o
recall subiu de **[CONFIRMAR]** para **[CONFIRMAR]** e a precisão caiu de
**[CONFIRMAR]** para **[CONFIRMAR]**: o modelo passa a indicar mais
publicações, encontrando mais peças realmente boas e também mais peças que não
mereciam. Como o número de espaços de divulgação é limitado e cada indicação
equivocada consome um espaço que não pode ser recuperado, optei pelo corte que
preserva a **[CONFIRMAR: precisão / recall]**, aceitando **[CONFIRMAR: perder
algumas peças promissoras / incluir algumas peças fracas]** em troca desse
ganho. Vale registrar que essas métricas foram obtidas em um único conjunto de
teste desta base sintética e podem variar com outra divisão dos dados.

---
## Antes de enviar

- [ ] Nome e matrícula no notebook
- [ ] Todas as células **executadas**, com os resultados visíveis
- [ ] Todo gráfico com título, eixos nomeados e a fonte na figura
- [ ] Todo `[CONFIRMAR]` substituído pelo número que saiu de fato
- [ ] Nenhuma afirmação de causalidade
- [ ] Baixar e abrir no VS Code para conferir antes de enviar